# Emotion-AI (DMESR) — live demo on Kaggle with a Cloudflare tunnel

This notebook downloads the app from GitHub, installs everything, pre-downloads the models, starts the
FastAPI backend and the React frontend, and opens a **free Cloudflare quick tunnel**. At the end you get a
public `https://….trycloudflare.com` link. Open it on any device to try the app; HTTPS lets the browser use
the microphone and camera.

**Before you click *Run All*:**
1. **Settings → Internet: On.** This needs a phone-verified Kaggle account.
2. **Settings → Accelerator: GPU T4** is recommended. CPU works, but replies are slower.
3. **Add-ons → Secrets.** Add `GROQ_API_KEY` (free at console.groq.com) and attach it to this notebook.
   Optional: `GROQ_API_KEY_FALLBACK1`, `GROQ_API_KEY_FALLBACK2`, `OPENAI_API_KEY`, `GEMINI_API_KEY`, and
   `GITHUB_TOKEN` if the repository is private.
   Secrets are **not** copied when someone copies this notebook, so every runner adds their own. You can also
   type a key in the app's Settings panel instead.

Setup takes about 5–10 minutes on first run. The demo link appears in step 10, and step 11 keeps it alive.

In [ ]:
# 1. Configuration
REPO_URL = "https://github.com/Vcode7/emotion-ai.git"
BRANCH = "main"

# /tmp is not saved with the notebook's outputs, so the code, keys and node_modules stay private.
WORK = "/tmp/emotion-ai"
TOOLS = "/tmp/emotion-ai-tools"
LOGS = "/tmp/emotion-ai-logs"
BACKEND_PORT = 8000
FRONTEND_PORT = 5173
NODE_LINE = "latest-v22.x"   # Node.js 22 LTS (Vite 6 needs Node 18+; Node 20 is end-of-life)

import hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

for d in (TOOLS, LOGS):
    os.makedirs(d, exist_ok=True)

_SECRET_RE = re.compile(r"(gsk_|sk-|AIza|ghp_|github_pat_)[A-Za-z0-9_\-]+")


def mask(text):
    """Hide anything that looks like an API key before printing."""
    return _SECRET_RE.sub(lambda m: m.group(1) + "***", text)


def run(cmd, cwd=None, env=None, show=True, display_cmd=None):
    """Run a shell command; stop Run All with a clear error if it fails."""
    print(f"$ {display_cmd or cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, cwd=cwd, env=env, text=True, capture_output=True)
    out = (p.stdout or "") + (p.stderr or "")
    if p.returncode != 0:
        print(mask(out[-4000:]))
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {display_cmd or cmd}")
    if show and out.strip():
        print(mask(out[-1500:]))
    return p.stdout


print("Python", sys.version.split()[0])

In [ ]:
# 2. Secrets (Kaggle Add-ons → Secrets). Values are never printed or written to disk.
APP_SECRET_NAMES = ["GROQ_API_KEY", "GROQ_API_KEY_FALLBACK1", "GROQ_API_KEY_FALLBACK2",
                    "OPENAI_API_KEY", "GEMINI_API_KEY"]
secrets = {}
try:
    from kaggle_secrets import UserSecretsClient
    _client = UserSecretsClient()
    for name in APP_SECRET_NAMES + ["GITHUB_TOKEN"]:
        try:
            value = _client.get_secret(name)
            if value and value.strip():
                secrets[name] = value.strip()
        except Exception:
            pass   # secret not added / not attached to this notebook
except ImportError:
    print("kaggle_secrets is unavailable (not on Kaggle?) — using environment variables only.")

for name in APP_SECRET_NAMES + ["GITHUB_TOKEN"]:
    if name not in secrets and os.environ.get(name):
        secrets[name] = os.environ[name].strip()

# The backend reads these from its environment (backend/config.py).
APP_ENV = {**os.environ, **{k: v for k, v in secrets.items() if k in APP_SECRET_NAMES}}

print("Secrets found:", ", ".join(sorted(secrets)) or "none")
if "GROQ_API_KEY" not in secrets:
    print("⚠ No GROQ_API_KEY secret. The app will start, but chat and Groq speech-to-text need a key: "
          "add the secret and re-run, or enter a key in the app's Settings panel.")

In [ ]:
# 3. Download the code from GitHub
token = secrets.get("GITHUB_TOKEN")
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(WORK, ".git")):
    run(f"git -C {WORK} pull --ff-only {auth_url} {BRANCH}",
        display_cmd=f"git -C {WORK} pull --ff-only {REPO_URL} {BRANCH}")
else:
    run(f"git clone --depth 1 -b {BRANCH} {auth_url} {WORK}",
        display_cmd=f"git clone --depth 1 -b {BRANCH} {REPO_URL} {WORK}")
    run(f"git -C {WORK} remote set-url origin {REPO_URL}", show=False)  # don't keep the token in .git/config

run(f"git -C {WORK} log -1 --oneline")

ckpt = Path(WORK, "model/checkpoint/dmesr_mosei_best.pt")
if not ckpt.exists() or ckpt.stat().st_size < 50_000_000:
    raise RuntimeError(f"DMESR checkpoint missing or truncated: {ckpt}")
for asset in ["frontend/public/models/sky.glb", "frontend/public/models/sky2.glb"]:
    if not Path(WORK, asset).exists():
        print(f"⚠ Avatar asset missing from the repo: {asset} (the 3D avatar will not load)")
print("Code ready at", WORK)

In [ ]:
# 4. System tools: ffmpeg, Node.js, cloudflared
if shutil.which("ffmpeg") is None:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null", show=False)
run("ffmpeg -version | head -1")

# Node.js 22 LTS — the checksum is verified against nodejs.org's SHASUMS256.txt
node_base = f"https://nodejs.org/dist/{NODE_LINE}/"
shasums = urllib.request.urlopen(node_base + "SHASUMS256.txt", timeout=60).read().decode()
m = re.search(r"^([0-9a-f]{64})\s+(node-v[\d.]+-linux-x64\.tar\.xz)$", shasums, re.M)
if not m:
    raise RuntimeError("Could not find the Linux x64 Node.js build in SHASUMS256.txt")
node_sha, node_file = m.groups()
node_dir = Path(TOOLS, node_file.replace(".tar.xz", ""))
if not (node_dir / "bin" / "node").exists():
    archive = Path(TOOLS, node_file)
    urllib.request.urlretrieve(node_base + node_file, archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != node_sha:
        raise RuntimeError("Node.js download checksum mismatch")
    run(f"tar -xJf {archive} -C {TOOLS}", show=False)
    archive.unlink()
os.environ["PATH"] = f"{node_dir}/bin:" + os.environ["PATH"]
APP_ENV["PATH"] = os.environ["PATH"]
run("node --version && npm --version")

# Cloudflare tunnel client (official release binary)
CLOUDFLARED = f"{TOOLS}/cloudflared"
if not os.path.exists(CLOUDFLARED):
    run(f"curl -fsSL -o {CLOUDFLARED} "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        f" && chmod +x {CLOUDFLARED}")
run(f"{CLOUDFLARED} --version")

In [ ]:
# 5. Python packages
# Keep Kaggle's preinstalled CUDA builds of torch / torchvision / torchaudio, and skip
# opencv-python-headless when cv2 already imports (two OpenCV wheels conflict).
skip = {"torch", "torchvision", "torchaudio"}
try:
    import cv2  # noqa: F401
    skip.add("opencv-python-headless")
except ImportError:
    pass

reqs = []
for line in Path(WORK, "backend/requirements.txt").read_text().splitlines():
    name = re.split(r"[<>=\[ ;]", line.strip(), maxsplit=1)[0].lower()
    if line.strip() and not line.startswith("#") and name not in skip:
        reqs.append(line.strip())
Path(TOOLS, "requirements-kaggle.txt").write_text("\n".join(reqs) + "\n")
run(f"{sys.executable} -m pip install -q -r {TOOLS}/requirements-kaggle.txt", show=False)

run(f"{sys.executable} -c \"import torch, torchaudio, transformers, fastapi, whisper; "
    "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
    "'| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__, "
    "'| fastapi', fastapi.__version__)\"")

In [ ]:
# 6. Pre-download every model, so the server starts fast and failures show up here
warmup = r"""
import whisper
from transformers import (AutoFeatureExtractor, AutoImageProcessor, AutoModelForAudioClassification,
                          AutoModelForSequenceClassification, AutoTokenizer, BertModel, BertTokenizer,
                          ViTForImageClassification, ViTImageProcessor)

whisper.load_model("base", device="cpu");                                   print("ok  whisper base")
BertTokenizer.from_pretrained("bert-base-uncased")
BertModel.from_pretrained("bert-base-uncased");                             print("ok  bert-base-uncased (DMESR)")
AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")
AutoModelForSequenceClassification.from_pretrained("SamLowe/roberta-base-go_emotions"); print("ok  text emotion")
AutoFeatureExtractor.from_pretrained("superb/wav2vec2-base-superb-er")
AutoModelForAudioClassification.from_pretrained("superb/wav2vec2-base-superb-er");      print("ok  voice emotion")
try:
    ViTImageProcessor.from_pretrained("HardlyHumans/Facial-expression-detection")
except Exception:
    AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")      # backend's fallback
ViTForImageClassification.from_pretrained("HardlyHumans/Facial-expression-detection"); print("ok  face emotion")
"""
Path(TOOLS, "warmup_models.py").write_text(warmup)
# Separate process so the notebook kernel does not hold GPU / RAM afterwards.
run(f"{sys.executable} {TOOLS}/warmup_models.py", cwd=f"{WORK}/backend", env=APP_ENV)

In [ ]:
# 7. Build the frontend
run("npm ci --no-audit --no-fund", cwd=f"{WORK}/frontend", env=APP_ENV, show=False)
run("npm run build", cwd=f"{WORK}/frontend", env=APP_ENV)

In [ ]:
# 8. Start the backend (FastAPI on 127.0.0.1:8000)
procs = globals().get("procs", {})


def start(name, cmd, cwd):
    old = procs.get(name)
    if old is not None and old.poll() is None:
        print(f"{name} already running (pid {old.pid})")
        return old
    log = open(f"{LOGS}/{name}.log", "w")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=APP_ENV, stdout=log, stderr=subprocess.STDOUT,
                         start_new_session=True)
    procs[name] = p
    print(f"started {name} (pid {p.pid}) — log: {LOGS}/{name}.log")
    return p


def tail(name, n=40):
    path = f"{LOGS}/{name}.log"
    if os.path.exists(path):
        lines = Path(path).read_text(errors="replace").splitlines()[-n:]
        print(f"── last {len(lines)} lines of {name}.log ──")
        print(mask("\n".join(lines)))


def get(url, timeout=10):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (emotion-ai notebook)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read().decode(errors="replace")


def wait_http(url, timeout_s, proc_name):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if procs[proc_name].poll() is not None:
            tail(proc_name, 80)
            raise RuntimeError(f"{proc_name} exited with code {procs[proc_name].returncode}")
        try:
            status, body = get(url, timeout=5)
            if status == 200:
                return body
        except Exception:
            pass
        time.sleep(3)
    tail(proc_name, 80)
    raise TimeoutError(f"{url} did not respond within {timeout_s}s")


start("backend", f"{sys.executable} -m uvicorn main:app --host 127.0.0.1 --port {BACKEND_PORT}",
      f"{WORK}/backend")
print("Loading models (can take a few minutes)…", flush=True)
print(wait_http(f"http://127.0.0.1:{BACKEND_PORT}/health", 600, "backend"))
print(json.dumps(json.loads(get(f"http://127.0.0.1:{BACKEND_PORT}/health/models")[1]), indent=2))

In [ ]:
# 9. Start the frontend (Vite preview on 127.0.0.1:5173). /api, /generated, /uploads and /static are proxied to the backend.
start("frontend", f"npx vite preview --host 127.0.0.1 --port {FRONTEND_PORT} --strictPort", f"{WORK}/frontend")
wait_http(f"http://127.0.0.1:{FRONTEND_PORT}/", 120, "frontend")
print("frontend ok; proxied backend health:", wait_http(f"http://127.0.0.1:{FRONTEND_PORT}/api/health", 60, "frontend"))

In [ ]:
# 10. Cloudflare quick tunnel → public HTTPS demo URL (no Cloudflare account needed)
from IPython.display import HTML, display

start("cloudflared", f"{CLOUDFLARED} tunnel --no-autoupdate --url http://127.0.0.1:{FRONTEND_PORT}", WORK)
DEMO_URL = None
deadline = time.time() + 120
while DEMO_URL is None and time.time() < deadline:
    if procs["cloudflared"].poll() is not None:
        tail("cloudflared", 60)
        raise RuntimeError("cloudflared exited before the tunnel was ready")
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", Path(LOGS, "cloudflared.log").read_text(errors="replace"))
    if found:
        DEMO_URL = found.group(0)
    else:
        time.sleep(1)
if DEMO_URL is None:
    tail("cloudflared", 60)
    raise TimeoutError("No trycloudflare.com URL within 120 s")

display(HTML(f'<h2>🎉 Demo URL: <a href="{DEMO_URL}" target="_blank">{DEMO_URL}</a></h2>'
             '<p>The link can take ~30 s to start working. Keep this session running (step 11).</p>'))
print("Demo URL:", DEMO_URL)

In [ ]:
# 11. Keep the demo alive and monitor it. Interrupt this cell (■) to stop everything.
from IPython.display import clear_output


def stop_all():
    for name, p in procs.items():
        if p.poll() is None:
            p.terminate()
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                p.kill()
            print(f"stopped {name}")


try:
    while True:
        states = {n: ("running" if p.poll() is None else f"exited ({p.returncode})") for n, p in procs.items()}
        try:
            status, _ = get(f"{DEMO_URL}/api/health", timeout=20)
            public = f"HTTP {status}"
        except Exception as exc:
            public = f"not reachable yet ({type(exc).__name__})"
        clear_output(wait=True)
        display(HTML(f'<h3>Demo URL: <a href="{DEMO_URL}" target="_blank">{DEMO_URL}</a></h3>'))
        print(time.strftime("%H:%M:%S"), "|", " | ".join(f"{n}: {s}" for n, s in states.items()),
              "| public /api/health:", public, flush=True)
        if any(p.poll() is not None for p in procs.values()):
            for n in procs:
                tail(n, 30)
            print("A process stopped — fix the error above and re-run steps 8–11.")
            break
        time.sleep(60)
except KeyboardInterrupt:
    stop_all()

## Using the demo

* Open the **Demo URL** from step 10 in Chrome or Edge, on a laptop or a phone.
* **Visual Chat**: click *Start Call* and allow the microphone and camera. Speak naturally. The panel shows the
  text, voice, face and DMESR-sentiment reads plus the fused **Overall** emotion. The avatar answers with voice.
* **Chat tab**: type or record messages. The assistant remembers the conversation per session
  (*Reset Session* starts a new one).

## Limits and troubleshooting

* The URL **changes on every run** and stops working when this Kaggle session ends (idle timeout or session limit).
  Quick tunnels are meant for testing and have no uptime guarantee.
* **`git clone` fails** — the repo is private (add a `GITHUB_TOKEN` secret) or `BRANCH` is wrong.
* **Downloads fail in steps 4–6** — Internet is off in the notebook settings.
* **Chat replies say "LLM error" / "No API key"** — add the `GROQ_API_KEY` secret and re-run from step 2,
  or enter a key in the app's Settings panel.
* **Page says "Blocked request. This host is not allowed"** — the repo's `frontend/vite.config.js` must list
  `.trycloudflare.com` in `preview.allowedHosts`.
* Logs: the `tail("backend")`, `tail("frontend")` and `tail("cloudflared")` helpers (files in `/tmp/emotion-ai-logs`).